In [16]:
from typing_extensions import TypedDict, Literal, Annotated
from typing import List
from langgraph.types import Send
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel
from operator import add

llm = init_chat_model("openai:gpt-4o-mini")
greeting_llm = llm.invoke("준비 됬니?")

print(greeting_llm.content)

네, 준비되었습니다! 무엇을 도와드릴까요?


In [17]:
class State(TypedDict) :
    document : str
    final_summary : str
    summaries : Annotated[list[dict], add]          # dictionary 배열


In [18]:
# 문장 요약 작업자
def worker_summarize_paragraph(args) : 
    paragraph = args["paragraph"]
    index = args["index"]

    res = llm.invoke(
        f"Write a 3-sentence summary for this paragraph : {paragraph}"
    )

    print(f""" [{index}] {res.content} """)

    return {
        "summaries" : [
            {
                "summary" : res.content,
                "index" : index
            }
        ]
    }

# worker 노드 실행자
def dispatch_summarizers(state : State) : 
    chunks = state["document"].split("\n\n")            # 줄바꿈 단위로 장문 쪼개기

    return[
        Send("worker_summarize_paragraph", {"paragraph" : chunk, "index" : index})
        for index, chunk in enumerate(chunks)
    ]

# 최종 요약
def final_summary(state : State):
    res = llm.invoke(
        f"Using the following summaries, give me a final one {state["summaries"]}"
    )    

    return {
        "final_summary" : res.content
    }


In [19]:

graph_builder = StateGraph(State)

graph_builder.add_node("worker_summarize_paragraph", worker_summarize_paragraph)
graph_builder.add_node("final_summary", final_summary)

graph_builder.add_conditional_edges(START, dispatch_summarizers, ["worker_summarize_paragraph"])
graph_builder.add_edge("worker_summarize_paragraph", "final_summary")
graph_builder.add_edge("final_summary", END)

graph = graph_builder.compile()


In [20]:
with open("fed_transcript.md", "r", encoding="utf-8") as file:
    document = file.read()

for chunk in graph.stream({"document" : document}, stream_mode="updates"):
    print(chunk, "\n")



 [2] The recent moderation and growth are primarily driven by a decline in consumer spending. This slowdown indicates shifting economic conditions that could impact various sectors. Overall, the trend suggests consumers are becoming more cautious with their expenditures. 
{'worker_summarize_paragraph': {'summaries': [{'summary': 'The recent moderation and growth are primarily driven by a decline in consumer spending. This slowdown indicates shifting economic conditions that could impact various sectors. Overall, the trend suggests consumers are becoming more cautious with their expenditures.', 'index': 2}]}} 

 [25] The committee welcomed a new member today, continuing its tradition of unity in achieving its dual mandate goals. The members expressed a strong commitment to maintaining their independence. However, there were no additional details or updates provided. 
{'worker_summarize_paragraph': {'summaries': [{'summary': 'The committee welcomed a new member today, continuing its trad